<div style="background-color: #1b2838; padding: 25px; border-radius: 12px; border-left: 6px solid #4caf50; box-shadow: 0 4px 6px rgba(0,0,0,0.1);">
    <h1 style="color: #ffffff; margin-top: 0; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; font-size: 2.2em;">🐄 Dairy Mate: Mastitis Detection Model Training</h1>
    <h3 style="color: #a5b4fc; margin-bottom: 5px; font-family: 'Segoe UI', sans-serif; font-weight: normal;">Automated Udder Segmentation & Deep Learning Pipeline</h3>
    <hr style="border: 1px solid #2a3f5a; margin-top: 15px; margin-bottom: 15px;">
    <p style="color: #d1d5db; margin-bottom: 0; font-family: 'Segoe UI', sans-serif; line-height: 1.6;">
        This notebook implements the training pipeline for classifying cow udders as <strong>Healthy</strong> or <strong>Mastitis</strong>. 
        It leverages a custom PyTorch dataset loader using the metadata from <code>Dataset/annotations.json</code> (including udder bounding boxes), 
        performs on-the-fly cropping/resizing, applies data augmentations to combat class imbalance, and trains two architectures: 
        <strong>ResNet-50</strong> (for high-capacity learning) and <strong>MobileNet-V3-Large</strong> (for light-weight edge deployment). 
        It is fully configured for Kaggle's dual NVIDIA Tesla T4 GPUs using PyTorch's <code>DataParallel</code>.
    </p>
</div>

### 📦 1. Importing Libraries
We import standard scientific and deep learning libraries. We will also use `safetensors` for save and load operations.

In [ ]:
import os
import json
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision import models
from PIL import Image
from tqdm import tqdm
from safetensors.torch import save_model

# Check device availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"Number of GPUs available: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

### ⚙️ 2. Hyperparameters & Configuration
We set hyperparameters for our training run. Given our small dataset of 100 images, we will keep batch size small (16) and limit epochs to avoid extreme overfitting.

In [ ]:
num_epochs = 20
batch_size = 16
learning_rate = 0.0001
num_classes = 2
seed = 42

# Set random seeds for reproducibility
random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

### 🖼️ 3. Custom Dataset & Data Split
We define a custom PyTorch dataset `IndexedMastitisDataset` which reads our JSON annotations, loads the udder images, crops them to the segmented udder boundary box, and applies data transformations.

In [ ]:
class IndexedMastitisDataset(Dataset):
    def __init__(self, annotations, indices, transform=None, crop_to_udder=True):
        self.annotations = [annotations[i] for i in indices]
        self.transform = transform
        self.crop_to_udder = crop_to_udder
        
    def __len__(self):
        return len(self.annotations)
        
    def __getitem__(self, idx):
        item = self.annotations[idx]
        img_path = item['filepath']
        
        # Load image
        if not os.path.exists(img_path):
            # Fallback path correction if running inside folders
            base_name = os.path.basename(img_path)
            folder_name = "healthy_images" if item['class_id'] == 0 else "Mastitis_images"
            img_path = os.path.join("Dataset", folder_name, base_name)
            
        img = Image.open(img_path).convert('RGB')
        
        # Crop to udder if bounding box is valid
        if self.crop_to_udder and 'bbox' in item:
            x, y, w, h = item['bbox']
            if w > 0 and h > 0:
                img = img.crop((x, y, x + w, y + h))
                
        if self.transform:
            img = self.transform(img)
            
        label = item['class_id']
        return img, label

### 🔄 4. Data Transforms and DataLoader Setup
Since our dataset is small (75 Mastitis, 25 Healthy), we apply color jitter, random rotations, and flips on the training set to prevent model memorization. We keep validation images unaugmented.

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Find annotations path
annotations_path = "Dataset/annotations.json"
if not os.path.exists(annotations_path):
    annotations_path = "../Dataset/annotations.json"  # Fallback if working inside a subdirectory
    
with open(annotations_path, 'r') as f:
    annotations = json.load(f)

# Create train/val split (80% train, 20% validation)
indices = list(range(len(annotations)))
random.shuffle(indices)
split_idx = int(0.8 * len(annotations))
train_indices = indices[:split_idx]
val_indices = indices[split_idx:]

# Create datasets
train_dataset = IndexedMastitisDataset(annotations, train_indices, transform=train_transform)
val_dataset = IndexedMastitisDataset(annotations, val_indices, transform=val_transform)

# Data loaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

print(f"Dataset loaded: {len(train_dataset)} training samples, {len(val_dataset)} validation samples.")

### 🏗️ 5. Model Architecture Selection
We build a loader function to initialize **ResNet-50** and **MobileNet-V3-Large** architectures. We modify their classification heads to output 2 class probabilities. We also wrap them in `nn.DataParallel` if more than 1 GPU is available.

In [ ]:
def get_model(model_name, num_classes=2):
    if model_name == "resnet50":
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif model_name == "mobilenetv3":
        model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V1)
        model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_classes)
    else:
        raise ValueError(f"Unknown model name: {model_name}")
        
    model = model.to(device)
    
    # Multi-GPU support (compatible with Kaggle dual T4)
    if torch.cuda.device_count() > 1:
        print(f"Wrapping {model_name} in DataParallel with {torch.cuda.device_count()} GPUs!")
        model = nn.DataParallel(model)
        
    return model

### 🏃 6. Training Pipeline
We define our model training execution loop, plotting loss curves and saving the best performing model weights on validation accuracy in safetensors format.

In [ ]:
def train_model(model_name, epochs=20):
    print(f"\n{'='*40}\nTraining {model_name.upper()} model\n{'='*40}")
    
    model = get_model(model_name)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    best_accuracy = 0.0
    
    for epoch in range(epochs):
        # Training loop
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        
        progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", unit="batch")
        for images, labels in progress:
            images, labels = images.to(device), labels.to(device)
            labels = labels.long()
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()
            
            progress.set_postfix(loss=(running_loss / len(train_loader)), acc=(100 * correct_train / total_train))
            
        train_acc = 100 * correct_train / total_train
        epoch_loss = running_loss / len(train_loader)
        
        # Validation phase
        model.eval()
        correct_val = 0
        total_val = 0
        running_val_loss = 0.0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                labels = labels.long()
                
                outputs = model(images)
                val_loss = criterion(outputs, labels)
                running_val_loss += val_loss.item()
                
                _, predicted = torch.max(outputs, 1)
                total_val += labels.size(0)
                correct_val += (predicted == labels).sum().item()
                
        val_acc = 100 * correct_val / total_val
        val_loss_avg = running_val_loss / len(val_loader)
        
        print(f"Epoch [{epoch+1}/{epochs}] - Loss: {epoch_loss:.4f} - Train Acc: {train_acc:.2f}% | Val Loss: {val_loss_avg:.4f} - Val Acc: {val_acc:.2f}%")
        
        # Save best performing checkpoint
        if val_acc > best_accuracy:
            best_accuracy = val_acc
            os.makedirs("Model", exist_ok=True)
            
            # Extract weights if DataParallel
            raw_model = model.module if isinstance(model, nn.DataParallel) else model
            save_path = f"Model/{model_name}_mastitis.safetensors"
            save_model(raw_model, save_path)
            print(f"  ✓ Saved new best model weights ({val_acc:.2f}%) to {save_path}")
            
    print(f"\nTraining completed for {model_name}. Best validation accuracy achieved: {best_accuracy:.2f}%")

### 🚀 7. Execute Training
We run the training runs for both architectures.

In [ ]:
# Train MobileNet-V3-Large (efficiency baseline)
train_model("mobilenetv3", epochs=num_epochs)

# Train ResNet-50 (accuracy baseline)
train_model("resnet50", epochs=num_epochs)

### 🎉 Conclusion
Both models are trained and saved. You can load their `.safetensors` weight files directly in the `app.py` script to run inference on cow udders.